In [1]:
def base_at_offset(seq: str, offset: int) -> str:
    """
    Return the base at given offset (0-based) in seq.
    Raises IndexError if offset is out of range.
    """
    if offset < 0 or offset >= len(seq):
        raise IndexError(f"Offset {offset} out of range for sequence of length {len(seq)}")
    return seq[offset]


def all_suffixes(seq: str):
    """
    Return all non-empty suffixes of seq as a list.
    suffix i starts at position i and goes to the end.
    """
    return [seq[i:] for i in range(len(seq))]


def count_suffixes(seq: str) -> int:
    """
    Number of non-empty suffixes of seq.
    For a string of length n, this is n.
    """
    return len(seq)


def all_prefixes(seq: str):
    """
    Return all non-empty prefixes of seq as a list.
    """
    return [seq[:i] for i in range(1, len(seq) + 1)]


def all_substrings(seq: str):
    """
    Return all substrings of seq (non-empty).
    There are n*(n+1)/2 substrings for length n.
    """
    n = len(seq)
    return [seq[i:j] for i in range(n) for j in range(i + 1, n + 1)]

In [31]:
# Question1 & 2
from Bio import SeqIO

i = 0
length_dict = {}

for record in SeqIO.parse("dna2.fasta", "fasta"):
    i += 1
    length_dict[record.id] = len(record.seq)

# 找最大值
max_len = max(length_dict.values())
max_keys = [k for k, v in length_dict.items() if v == max_len]

# 找最小值
min_len = min(length_dict.values())
min_keys = [k for k, v in length_dict.items() if v == min_len]

print("Number of records:", i)
print("=== Longest sequence(s) ===")
print("Length:", max_len)
print("ID(s):", max_keys)

print("\n=== Shortest sequence(s) ===")
print("Length:", min_len)
print("ID(s):", min_keys)

Number of records: 18
=== Longest sequence(s) ===
Length: 4894
ID(s): ['gi|142022655|gb|EQ086233.1|255']

=== Shortest sequence(s) ===
Length: 115
ID(s): ['gi|142022655|gb|EQ086233.1|346']


In [21]:
# Question3
from Bio import SeqIO

# -----------------------------
# Helper: find ORFs in one sequence
# -----------------------------
def find_orfs(seq, frame):
    """
    Find all ORFs in a given sequence and reading frame.
    seq: string (DNA)
    frame: 1, 2, or 3
    Returns a list of tuples: (start_pos, end_pos, length)
    start_pos and end_pos are 1-based positions in the original sequence.
    """
    start_codon = "ATG"
    stop_codons = {"TAA", "TAG", "TGA"}

    orfs = []
    n = len(seq)

    # Convert frame (1,2,3) to Python index (0,1,2)
    start_index = frame - 1

    i = start_index
    while i + 3 <= n:
        codon = seq[i:i+3]

        # Found start codon
        if codon == start_codon:
            j = i + 3
            # Scan forward in same frame
            while j + 3 <= n:
                codon2 = seq[j:j+3]
                if codon2 in stop_codons:
                    # ORF found
                    start_pos = i + 1      # convert to 1-based
                    end_pos = j + 3        # inclusive, 1-based
                    length = end_pos - start_pos + 1
                    orfs.append((start_pos, end_pos, length))
                    break
                j += 3
        i += 3

    return orfs


# -----------------------------
# Main: scan FASTA file
# -----------------------------
def scan_fasta_for_orfs(fasta_path, frame):
    """
    Scan all sequences in a FASTA file for ORFs in the given frame.
    Returns:
        longest_orf_info: (seq_id, start_pos, end_pos, length)
        per_sequence_longest: {seq_id: (start_pos, end_pos, length)}
    """
    longest_orf_info = None
    per_sequence_longest = {}

    for record in SeqIO.parse(fasta_path, "fasta"):
        seq_id = record.id
        seq = str(record.seq)

        orfs = find_orfs(seq, frame)

        if orfs:
            # pick longest ORF in this sequence
            longest = max(orfs, key=lambda x: x[2])
            per_sequence_longest[seq_id] = longest

            # update global longest
            if (longest_orf_info is None) or (longest[2] > longest_orf_info[3]):
                longest_orf_info = (seq_id, longest[0], longest[1], longest[2])
        else:
            per_sequence_longest[seq_id] = None

    return longest_orf_info, per_sequence_longest


# -----------------------------
# Example usage
# -----------------------------
fasta_file = "dna2.fasta"
frame = 3  # choose 1, 2, or 3

longest_orf, per_seq = scan_fasta_for_orfs(fasta_file, frame)

print("=== Longest ORF in entire FASTA ===")
if longest_orf:
    seq_id, start, end, length = longest_orf
    print("Sequence ID:", seq_id)
    print("Start position:", start)
    print("End position:", end)
    print("Length:", length)
else:
    print("No ORFs found in this frame.")

print("\n=== Longest ORF per sequence ===")
for sid, info in per_seq.items():
    print(sid, "→", info)

=== Longest ORF in entire FASTA ===
Sequence ID: gi|142022655|gb|EQ086233.1|527
Start position: 636
End position: 2456
Length: 1821

=== Longest ORF per sequence ===
gi|142022655|gb|EQ086233.1|91 → (2856, 3443, 588)
gi|142022655|gb|EQ086233.1|304 → (621, 767, 147)
gi|142022655|gb|EQ086233.1|255 → (1641, 1925, 285)
gi|142022655|gb|EQ086233.1|45 → (699, 1022, 324)
gi|142022655|gb|EQ086233.1|396 → (429, 653, 225)
gi|142022655|gb|EQ086233.1|250 → (1374, 1622, 249)
gi|142022655|gb|EQ086233.1|322 → (90, 278, 189)
gi|142022655|gb|EQ086233.1|88 → None
gi|142022655|gb|EQ086233.1|594 → (66, 278, 213)
gi|142022655|gb|EQ086233.1|293 → (2334, 3044, 711)
gi|142022655|gb|EQ086233.1|75 → (57, 260, 204)
gi|142022655|gb|EQ086233.1|454 → (3096, 4496, 1401)
gi|142022655|gb|EQ086233.1|16 → (1440, 3083, 1644)
gi|142022655|gb|EQ086233.1|584 → (348, 479, 132)
gi|142022655|gb|EQ086233.1|4 → (1278, 1496, 219)
gi|142022655|gb|EQ086233.1|277 → (759, 929, 171)
gi|142022655|gb|EQ086233.1|346 → None
gi|142022655|gb|

In [34]:
# Question4
from Bio import SeqIO
from collections import defaultdict

def find_repeats_in_fasta(fasta_path, n):
    """
    Find all repeats of length n in all sequences in the FASTA file.
    Returns:
        repeats: dict {repeat: count}  # only repeats with count >= 2
        most_frequent: list of (repeat, count)  # all tied max repeats
    """
    counts = defaultdict(int)

    # Scan all sequences
    for record in SeqIO.parse(fasta_path, "fasta"):
        seq = str(record.seq)
        L = len(seq)

        # Extract all substrings of length n (allow overlaps)
        for i in range(L - n + 1):
            sub = seq[i:i+n]
            counts[sub] += 1

    # Only repeats (count >= 2)
    repeats = {k: v for k, v in counts.items() if v >= 2}

    if repeats:
        # Step 1: find the maximum count
        max_count = max(repeats.values())

        # Step 2: collect ALL repeats with that count
        most_frequent = [(k, v) for k, v in repeats.items() if v == max_count]
    else:
        most_frequent = []

    return repeats, most_frequent


# -----------------------------
# Example usage
# -----------------------------
fasta_file = "dna2.fasta"
n = 12  # repeat length

repeats, most_frequent = find_repeats_in_fasta(fasta_file, n)

print("\n=== Most frequent repeat ===")
print(most_frequent)
print("\n")
print("=== All repeats of length", n, "===")
for r, c in repeats.items():
    print(r, "→", c)


=== Most frequent repeat ===
[('CATTCGCCATTC', 10), ('ATTCGCCATTCG', 10), ('TTCGCCATTCGC', 10), ('TCGCCATTCGCC', 10)]


=== All repeats of length 12 ===
CGGCGTGTCGCG → 2
TCGTCGGCGATC → 2
CGGCGCGAACAC → 2
GGCGCGAACACG → 2
GCGCGAACACGC → 2
CGCGAACACGCC → 2
CGAACACGCCGG → 2
GAACACGCCGGC → 2
CACGCCGGCCGG → 2
GGCGGCCTGTGG → 2
GCGGCCTGTGGG → 2
CGCGTTCAGCGC → 3
GCGTTCAGCGCC → 2
CGTTCAGCGCCG → 2
TACAACGTCGGG → 2
TGTCGGACGTGC → 2
GCGGCGGCTTCC → 2
CGGCTTCCAGTA → 2
GGCTTCCAGTAC → 2
GCTTCCAGTACC → 2
AGCAGCGCGGCG → 2
GCAGCGCGGCGG → 2
CAGCGCGGCGGC → 2
GCGGCGACATCG → 2
CAACCAGGTCAC → 2
AACCAGGTCACG → 2
CACGCGGGCACG → 2
GGCGGCGTCCGG → 2
GCGGCGTCCGGC → 3
TCTGACGATGCG → 2
CTGACGATGCGC → 2
TGCGGCCCGACG → 2
GCCGAGCACGTG → 2
CCGAGCACGTGC → 2
CGAGCATCATCG → 2
GAGCATCATCGA → 2
AGCATCATCGAC → 2
TTGGTCACGTCG → 2
CTGCTGCGCGCG → 3
TGCTGCGCGCGG → 2
GCTGCGCGCGGC → 2
CGCGGTTCCGCG → 2
GGTGCGCGCCGA → 2
TCGCCGAGCCAG → 2
CGAGCCAGTTGC → 2
CTGCCGAACATC → 2
GACGTCACGCCG → 2
CGTCACGCCGAG → 2
TCACGCCGAGCG → 2
CGAGCGCGCCGG 